# מעבדה 15 — מעברי פאזה ומודל איזינג

במעבדה הזו אתם בונים סריג של ספינים, צופים בו מסתדר ומאבד את הסדר, ואז מריצים את הניסויים
שהמודול בנוי עליהם. אתם בודקים את הסימולציה מול הפתרון המדויק של השרשרת החד-ממדית לפני שאתם
סומכים עליה בכל דבר אחר; סורקים את הסריג הריבועי דרך המעבר שלו בשלושה גדלים, וקוראים את
הסוספטיביליות ואת קיבול החום מתוך הפלוקטואציות; מודדים כמה זמן לוקח לסימולציה לשכוח את עצמה,
והיכן הזמן הזה מגיע לשיא; צופים בסריג קטן קופץ בין שני מצביו הממוגנטים בעוד סריג גדול יותר
מסרב לכך; סורקים שדה למעלה ולמטה, מתחת למעבר ומעליו; ומריצים את אותה פיזיקה תחת שני כללי עדכון
שונים. אתם מסיימים במדידה של טמפרטורת המעבר, עם אי-הוודאות שלה מוצהרת ביושר.

המסלול: האובייקטים (חלק 1); מבט ראשון בסריג (חלק 2); השרשרת החד-ממדית מול הפתרון המדויק שלה
(חלק 3); סריקת טמפרטורה בשלושה גדלים (חלק 4); האטה קריטית (חלק 5); מי בוחר את הכיוון (חלק 6);
היסטרזיס (חלק 7); שני כללי עדכון, שיווי משקל אחד (חלק 8); מדידה של $T_c$ (חלק 9); הבדיקות
האוטומטיות (חלק 10); וחקירה חופשית (חלק 11).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

**יחידות.** אנרגיות נמדדות ביחידות של הצימוד $J$, טמפרטורות ביחידות של $J/\kB$ ושדות ביחידות
של $J$, כך ש-`T = 2.0` פירושו $\kB T = 2J$. המעבר המדויק של הסריג הריבועי נמצא ב-$2.269$.

## מפרט המודל

| | |
|---|---|
| **מערכת** | סריג ריבועי $L \times L$ של ספינים קלאסיים $s_i = \pm 1$, צימוד בין שכנים קרובים $J > 0$, שדה אחיד $h$; בחלק 3, טבעת של ספינים |
| **דינמיקה** | שרשרת מרקוב, לא משוואות תנועה: חצאי סריקות בעדכון לוח שחמט עם כלל מטרופוליס $\min(1, e^{-\Delta E/(\kB T)})$, או, במקום שהדבר נאמר, כלל גלאובר; סריקות אינן זמן פיזיקלי |
| **גבול** | מחזורי; $T$ ו-$h$ נכפים על ידי אמבט חום מובלע |
| **צבר** | קנוני ב-$T$: ההתפלגות הסטציונרית של השרשרת, מכוח איזון מפורט |
| **מוזנח** | ספין קוונטי, תנודות הסריג, כוחות דיפוליים ארוכי טווח, אנאיזוטרופיה, אי-סדר, היתפסות של קירות תחום |
| **תקף כאשר** | השרשרת רצה מעבר לשלב החימום שלה ולאורך זמני מתאם עצמי רבים $\tau(T)$ |
| **אופני כישלון** | ליד $T_c$ הזמן $\tau$ גדל עם $L$ ופסי שגיאה נאיביים הם אשליה; מתחת ל-$T_c$ ובשדה $|h|$ קטן, ריצה נשארת בענף מגנוט אחד; עקבה ששומרת את שלב החימום שלה מוטה |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import ising
from thermolab.validation import relative_error, seed_study

# Every stochastic call below takes an explicit generator; nothing touches global random
# state, so re-running a cell reproduces it exactly.
rng = np.random.default_rng(15)

T_C = ising.onsager_tc()
print(f"Onsager's exact T_c = {T_C:.6f} J/k_B;  mean field says {ising.mean_field_tc():.0f}")

## חלק 1 — האובייקטים

| פונקציה | מה היא נותנת |
|---|---|
| `random_state(L, rng)`, `aligned_state(L)` | סריג $L \times L$ חם (אקראי) או קר (כולו למעלה); `dimension=1` נותן טבעת |
| `simulate(state, T, n, rng, burn_in, thin, rule)` | מריצה את השרשרת: משליכה `burn_in` סריקות, ואז רושמת את $m$ ואת $e = E/N$ כל `thin` סריקות; מחזירה את שתי העקבות ואת המצב הסופי |
| `observables(m, e, T, N)` | $\langle |m| \rangle$ ו-$\langle e \rangle$ עם שגיאות מתוקנות לפי $\tau$, וכן $\chi' = N\,\mathrm{Var}(|m|)/T$, $c = N\,\mathrm{Var}(e)/T^2$, ו-$\tau$ של כל עקבה |
| `temperature_scan(L, temps, n, rng)` | `observables` לאורך סולם של טמפרטורות |
| `autocorrelation_time(trace)`, `mean_with_error(trace)` | $\tau$ אינטגרלי; ממוצע, שגיאת תקן מתוך $N_{\mathrm{eff}} = n/(2\tau)$, ו-$\tau$ |
| `hysteresis_sweep(state, T, fields, sweeps, rng)`, `hysteresis_fields(h_max, n)`, `loop_area` | $m$ לאורך סולם של שדות, בכיוון $+h \to -h \to +h$, ושטח הלולאה |
| `branch_flips(m)` | כמה פעמים עקבה עוברת בין $m > 0.5$ לבין $m < -0.5$ |
| `ising_1d_exact(T, h)` | הזוג $(e, m)$ המדויק של הטבעת האינסופית |
| `onsager_tc()`, `onsager_magnetization(T)` | $T_c$ המדויק של הסריג הריבועי והמגנוט הספונטני שלו |
| `mean_field_magnetization(T, h)` | הפתרון של $m = \tanh((4m + h)/T)$ |

המצב מעדכן את האנרגיה שלו ואת הספין הכולל שלו היפוך אחר היפוך; חלק 10 בודק שהחשבון הזה לעולם
אינו סוטה מספירה מחדש.

### לנבא

התחייבו לתשובה לכל אחת מהשאלות לפני שתריצו משהו.

1. סריג $64 \times 64$ ב-$T = 1.5$, שמורץ מהתחלה אקראית ומהתחלה שכולה למעלה: האם שתי הריצות
   מסתיימות באותו $|m|$? ומה ב-$T = 3.5$?
2. באיזו טמפרטורה לוקח לסימולציה הכי הרבה זמן לאבד את המתאם: $1.5$, $2.27$ או $3.5$?
3. האם $\langle |m| \rangle(T)$ תלולה יותר במעבר עבור $L = 8$ או עבור $L = 32$?
4. מתחת ל-$T_c$ בשדה אפס, האם סריג $8 \times 8$ שומר על סימן המגנוט שלו לאורך ריצה ארוכה?
   ומה סריג $32 \times 32$?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — מבט ראשון בסריג

הריצו סריג $64 \times 64$ במשך 400 סריקות ב-$T = 1.5$ וב-$T = 3.5$, כל אחת מהתחלה חמה ומהתחלה
קרה, והתבוננו בתצורות הסופיות ובעקבות המגנוט.

In [ ]:
first_look = {}
fig, axes = plt.subplots(2, 4, figsize=(13, 6), gridspec_kw={"height_ratios": [1.0, 0.6]})
for k, (temperature, start) in enumerate([(1.5, "hot"), (1.5, "cold"), (3.5, "hot"),
                                         (3.5, "cold")]):
    state = ising.random_state(64, rng) if start == "hot" else ising.aligned_state(64)
    m, e, state = ising.simulate(state, temperature, 400, rng, burn_in=0)
    first_look[(temperature, start)] = m
    axes[0, k].imshow(state.lattice, cmap="Blues", vmin=-1.5, vmax=1)
    axes[0, k].set_title(f"T = {temperature}, {start} start")
    axes[0, k].set_xticks([])
    axes[0, k].set_yticks([])
    axes[1, k].plot(m, lw=1)
    axes[1, k].set_ylim(-1.05, 1.05)
    axes[1, k].set_xlabel("sweep")
axes[1, 0].set_ylabel("m")
plt.tight_layout()
plt.show()

for (temperature, start), m in first_look.items():
    print(f"T = {temperature}, {start:>4} start: m over the last 100 sweeps = "
          f"{m[-100:].mean():+.3f}")
print(f"Onsager-Yang |m| at T = 1.5: {ising.onsager_magnetization(1.5):.4f}")

ב-$T = 3.5$ שתי ההתחלות שוכחות מאין באו בתוך סריקות ספורות ומשוטטות סביב
$m = 0$. ב-$T = 1.5$ ההתחלה הקרה נשארת ליד $0.987$; ההתחלה החמה היא טלאים של תחומים
שעדיין עוברים הגסה אחרי 400 סריקות — ולפעמים נתקעת בשני פסים לזמן ארוך הרבה יותר. *אף אחד*
מהם אינו ראיה לכך ששיווי המשקל ב-$1.5$ הוא חסר סדר: זו ראיה לכך שההגעה אליו מהתחלה אקראית
איטית.

## חלק 3 — קודם מכיילים: השרשרת מול הפתרון המדויק שלה

סימולציה זוכה באמון כשהיא משחזרת משהו שידוע במדויק. לטבעת יש פתרון מדויק (שלב 3 של המודול):
ב-$h = 0$, $e = -\tanh(1/T)$ ו-$m = 0$; ובשדה,
$m = \sinh(h/T)/\sqrt{\sinh^2(h/T) + e^{-4/T}}$. הריצו טבעת של 1024 אתרים תחת ארבעה זרעים בלתי
תלויים בכל אחת משלוש נקודות והשוו. הריצות האלה משתמשות בכלל גלאובר, מפני שבממד אחד
ב-$h = 0$ אלגוריתם מטרופוליס מזיז קירות תחום באופן דטרמיניסטי תחת עדכון לוח השחמט ושוכח את
נקודת ההתחלה שלו לאט — ה-docstring של המודול `ising` מביא את המדידה.

### לנבא

האם האנרגיות המדומות יסכימו עם המדויקות בתוך שלוש שגיאות תקן? מה המשמעות אם לא?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
chain_points = [(1.0, 0.0), (2.0, 0.0), (1.0, 0.3)]
chain = {}
for temperature, field in chain_points:
    def run(gen, temperature=temperature, field=field):
        state = ising.random_state(1024, gen, field=field, dimension=1)
        m, e, _ = ising.simulate(state, temperature, 500, gen, burn_in=300, rule="glauber")
        return e.mean(), m.mean()

    results = np.array([run(np.random.default_rng(s))
                        for s in np.random.SeedSequence(3).spawn(4)])
    chain[(temperature, field)] = results
    e_exact, m_exact = ising.ising_1d_exact(temperature, field)
    e_se = results[:, 0].std(ddof=1) / 2
    m_se = results[:, 1].std(ddof=1) / 2
    print(f"T = {temperature}, h = {field}:  e = {results[:, 0].mean():.4f} +/- {e_se:.4f} "
          f"(exact {e_exact:.4f}, {abs(results[:, 0].mean() - e_exact) / e_se:.1f} sigma);  "
          f"m = {results[:, 1].mean():+.4f} +/- {m_se:.4f} (exact {m_exact:+.4f})")

t_line = np.linspace(0.3, 4.0, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_line, ising.ising_1d_exact(t_line)[0], color="k", label="exact, h = 0")
ax.plot(t_line, ising.ising_1d_exact(t_line, 0.3)[0], color="grey", ls="--",
        label="exact, h = 0.3")
for (temperature, field), results in chain.items():
    ax.errorbar(temperature, results[:, 0].mean(), yerr=3 * results[:, 0].std(ddof=1) / 2,
                fmt="o", color="#dc2626" if field else "#2563eb", capsize=3)
ax.set_xlabel("k_B T / J")
ax.set_ylabel("E / (N J)")
ax.legend()
plt.tight_layout()
plt.show()

הסימולציה משחזרת את השרשרת המדויקת בתוך פסי השגיאה שלה. עכשיו היא הרוויחה את הזכות שישאלו
אותה על הסריג הריבועי, שאין לו פתרון מדויק לרוב מה שנמדוד.

## חלק 4 — סריקת טמפרטורה בשלושה גדלים

סרקו את הסריג הריבועי מ-$T = 3.3$ ומטה עד $1.8$ עבור $L = 8$, $16$ ו-$32$, עם רשת טמפרטורות
צפופה ליד $2.3$. בכל טמפרטורה השליכו 300 סריקות ושמרו 1200. מתוך העקבות: $\langle |m| \rangle$,
הסוספטיביליות $\chi' = N\,\mathrm{Var}(|m|)/T$ וקיבול החום לספין $c = N\,\mathrm{Var}(e)/T^2$ —
זהויות הפלוקטואציה של מודול 12, עם $|m|$ במקום $m$ מהסיבה ששלב 6 של המודול נותן.

זה התא הארוך ביותר במעבדה: בסך הכול כ-$10^5$ סריקות. בדפדפן צפו לדקה או שתיים.

### לנבא

שרטטו את $\langle|m|\rangle(T)$ לשלושת הגדלים על גרף אחד לפני ההרצה. היכן $\chi'$
יגיע לשיא, והאם השיא יזוז כאשר $L$ גדל?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
SIZES = (8, 16, 32)
temps = np.array([3.3, 3.0, 2.8, 2.65, 2.55, 2.48, 2.42, 2.37, 2.32, 2.28, 2.24, 2.2, 2.1, 1.95,
                  1.8])
scans = {}
for size in SIZES:
    # temperature_scan walks the ladder in the order given, starting each T from the last
    # configuration; cooling from hot avoids starting in an ordered state above T_c.
    scans[size] = ising.temperature_scan(size, temps, 1200, rng, burn_in=300)


def column(size, name):
    return np.array([getattr(r, name) for r in scans[size]])


colours = {8: "#059669", 16: "#d97706", 32: "#dc2626"}
t_fine = np.linspace(1.75, 3.35, 400)
fig, (ax_m, ax_chi, ax_c) = plt.subplots(1, 3, figsize=(14, 4))
ax_m.plot(t_fine, ising.onsager_magnetization(t_fine), color="k", lw=1.2,
          label="Onsager, L = infinity")
ax_m.plot(t_fine, ising.mean_field_magnetization(t_fine), color="grey", ls=":",
          label="mean field")
for size in SIZES:
    ax_m.errorbar(temps, column(size, "abs_magnetization"),
                  yerr=column(size, "abs_magnetization_error"), fmt="o-", ms=3,
                  color=colours[size], label=f"L = {size}")
    ax_chi.plot(temps, column(size, "susceptibility"), "o-", ms=3, color=colours[size])
    ax_c.plot(temps, column(size, "heat_capacity"), "o-", ms=3, color=colours[size])
ax_m.set_ylabel("<|m|>")
ax_chi.set_ylabel("chi' (per spin)")
ax_c.set_ylabel("c (per spin, units of k_B)")
for ax in (ax_m, ax_chi, ax_c):
    ax.axvline(T_C, color="grey", ls="--", lw=1)
    ax.set_xlabel("k_B T / J")
ax_m.legend(fontsize=8)
plt.tight_layout()
plt.show()

peak_t = {size: temps[np.argmax(column(size, "susceptibility"))] for size in SIZES}
for size in SIZES:
    chi = column(size, "susceptibility")
    print(f"L = {size:2d}: chi' peaks at T = {peak_t[size]:.2f} with height {chi.max():6.2f};  "
          f"c peaks at T = {temps[np.argmax(column(size, 'heat_capacity'))]:.2f}")

שלושה דברים לקרוא מהגרפים. עקומות המגנוט הן כתפיים מעוגלות, חדות יותר עבור $L$ גדול יותר;
אף אחת מהן אינה נוגעת באפס, מפני שבסריג סופי $|m|$ לעולם אינו מתאפס. שיא הסוספטיביליות
גדל עם $L$ וזז *מטה* לעבר $T_c$. ועקומת השדה הממוצע אפילו אינה במקום הנכון: היא מסתדרת
ב-$4.0$.

## חלק 5 — האטה קריטית

כל רשומה של `observables` נושאת גם את זמן המתאם העצמי של סדרת $|m|$ שלה, בסריקות. שרטטו אותו
כנגד הטמפרטורה.

### לנבא

באיזו טמפרטורה $\tau$ יהיה הגדול ביותר? האם שם הוא יגדל או יקטן עם $L$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for size in SIZES:
    ax.plot(temps, column(size, "tau_abs_magnetization"), "o-", ms=4, color=colours[size],
            label=f"L = {size}")
ax.axvline(T_C, color="grey", ls="--", lw=1)
ax.set_yscale("log")
ax.set_xlabel("k_B T / J")
ax.set_ylabel("tau of |m| (sweeps)")
ax.legend()
plt.tight_layout()
plt.show()

for size in SIZES:
    tau = column(size, "tau_abs_magnetization")
    print(f"L = {size:2d}: tau = {tau[0]:.1f} at T = {temps[0]}, {tau.max():.1f} at "
          f"T = {temps[np.argmax(tau)]}, {tau[-1]:.1f} at T = {temps[-1]}")
tau_peak = {size: column(size, "tau_abs_magnetization").max() for size in SIZES}

הסימולציה איטית ביותר בדיוק במעבר, והשיא גדל עם הסריג: זו האטה קריטית. עמוק בתוך כל אחת
מהפאזות $|m|$ מאבד את המתאם בתוך סריקה או שתיים. כל פס שגיאה בחלק 4 כבר מביא זאת בחשבון —
$\sigma/\sqrt{N_{\mathrm{eff}}}$ עם $N_{\mathrm{eff}} = n/(2\tau)$ — ולכן הנקודות ליד $T_c$
נושאות את הפסים הגדולים ביותר.

## חלק 6 — מי בוחר את הכיוון?

מתחת ל-$T_c$ בשדה אפס הממוצע הקנוני של $m$ הוא אפס בדיוק, מטעמי סימטריה. הריצו סריג
$8 \times 8$ וסריג $32 \times 32$ ב-$T = 2.1$ במשך 6000 סריקות כל אחד, מהתחלה שכולה למעלה,
ועקבו אחר הסימן של $m$.

### לנבא

כמה פעמים כל סריג יהפוך את המגנוט שלו? מה יהיה הממוצע בזמן של $m$ עבור כל אחד, ומה של
$|m|$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
branch = {}
fig, ax = plt.subplots(figsize=(10, 3.5))
for size, colour in ((8, "#059669"), (32, "#dc2626")):
    m, _, _ = ising.simulate(ising.aligned_state(size), 2.1, 6000, rng, burn_in=0)
    branch[size] = m
    ax.plot(m, lw=0.8, color=colour, label=f"L = {size}")
ax.axhline(ising.onsager_magnetization(2.1), color="grey", ls="--", lw=1)
ax.axhline(-ising.onsager_magnetization(2.1), color="grey", ls="--", lw=1)
ax.set_ylim(-1.05, 1.05)
ax.set_xlabel("sweep")
ax.set_ylabel("m")
ax.legend()
plt.tight_layout()
plt.show()

for size, m in branch.items():
    print(f"L = {size:2d}: {ising.branch_flips(m):3d} reversals;  <m> = {m.mean():+.3f};  "
          f"<|m|> = {np.abs(m).mean():.3f}")
print(f"Onsager-Yang m_0(2.1) = {ising.onsager_magnetization(2.1):.4f}")

הסריג הקטן מתהפך שוב ושוב, והממוצע שלו בזמן נסחף לעבר אפס, התשובה של הצבר. הגדול לעולם אינו
מתהפך: הממוצע שלו בזמן הוא $+0.87$, המגנוט הספונטני, והוא היה נשאר שם הרבה מעבר לכל ריצה
שתוכלו להרשות לעצמכם. לשניהם אותו $\langle |m| \rangle$. הממוצע על הצבר הוא אפס; ה*דגימה*
אינה — וככל שהדגימה גדולה יותר, כך השניים חדלים באופן שלם יותר להיות אותו דבר.

## חלק 7 — היסטרזיס

סרקו את השדה $+1 \to -1 \to +1$ ב-81 צעדים, 10 סריקות לצעד, על סריג $32 \times 32$
ב-$T = 1.5$ (מתחת ל-$T_c$) וב-$T = 3.0$ (מעליה). אחר כך חזרו על הלולאה הקרה לאט יותר.

### לנבא

האם אחת מהלולאות תיסגר? איזו, אם תסרקו לאט יותר?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
fields = ising.hysteresis_fields(1.0, 41)
loops = {}
for temperature, per_step in ((1.5, 10), (1.5, 40), (3.0, 10)):
    h, m, _ = ising.hysteresis_sweep(ising.aligned_state(32), temperature, fields, per_step,
                                     rng)
    loops[(temperature, per_step)] = (h, m, ising.loop_area(h, m))

fig, ax = plt.subplots(figsize=(6, 4.5))
styles = {(1.5, 10): ("#2563eb", "-"), (1.5, 40): ("#2563eb", ":"), (3.0, 10): ("grey", "-")}
for key, (h, m, area) in loops.items():
    colour, ls = styles[key]
    ax.plot(h, m, color=colour, ls=ls, label=f"T = {key[0]}, {key[1]} sweeps/step: area {area:.2f}")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("h / J")
ax.set_ylabel("m")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for key, (h, m, area) in loops.items():
    at_zero = m[np.argmin(np.abs(h[:41]))]
    print(f"T = {key[0]}, {key[1]:2d} sweeps per step: loop area {area:.3f};  "
          f"m at h = 0 on the way down {at_zero:+.2f}")

מתחת ל-$T_c$ המגנוט שומר על הסימן שלו הרבה מעבר ל-$h = 0$ ואז מתהפך בבת אחת: הסריג
*מטא-יציב* — הנוזל המחומם-יתר של מודול 14, בגרסה מגנטית — עד שבמקרה נוצר תחום הפוך גדול דיו
כדי לגדול. סריקה איטית פי ארבעה מצרה את הלולאה אבל אינה סוגרת אותה. מעל $T_c$ העקומה עוברת
דרך הראשית וכמעט חוזרת על עצמה; מה שנותר מהשטח שלה הוא פיגור ורעש. לאורך $h = 0$ מתחת
ל-$T_c$ המגנוט קופץ בין $\pm m_0$: קו של מעבר מסדר ראשון, שנגמר בנקודה הקריטית.

## חלק 8 — שני כללי עדכון, שיווי משקל אחד

איזון מפורט קובע *לאן* השרשרת הולכת ואינו אומר דבר על המהירות. הריצו את אלגוריתם מטרופוליס
ואת כלל גלאובר זה לצד זה: תחילה אותו צינון חטוף (התחלה אקראית, $T = 1.5$, $32 \times 32$)
תחת שלושה זרעים לכל אחד, ואחר כך את אנרגיית שיווי המשקל ב-$T = 2.8$ תחת ארבעה זרעים לכל אחד.

### לנבא

איזה כלל מגיע לרלקסציה מהר יותר אחרי הצינון החטוף? האם הם יסכימו על אנרגיית שיווי המשקל
ב-$2.8$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
relax = {}
for rule in ("metropolis", "glauber"):
    traces = [ising.simulate(ising.random_state(32, np.random.default_rng(s)), 1.5, 150,
                             np.random.default_rng(s), burn_in=0, rule=rule)[1]
              for s in np.random.SeedSequence(8).spawn(3)]
    relax[rule] = np.mean(traces, axis=0)


def equilibrium_energy(rule):
    def measure(gen):
        _, e, _ = ising.simulate(ising.random_state(16, gen), 2.8, 900, gen, burn_in=200,
                                 rule=rule)
        return float(e.mean())
    return seed_study(measure, n_seeds=4)


equilibrium = {rule: equilibrium_energy(rule) for rule in ("metropolis", "glauber")}

fig, ax = plt.subplots(figsize=(7, 4))
for rule, colour in (("metropolis", "#2563eb"), ("glauber", "#dc2626")):
    ax.plot(np.arange(1, 151), relax[rule], color=colour, label=rule)
ax.set_xscale("log")
ax.set_xlabel("sweep")
ax.set_ylabel("E / (N J) after a quench to T = 1.5")
ax.legend()
plt.tight_layout()
plt.show()

for rule in relax:
    study = equilibrium[rule]
    print(f"{rule:>10}: e after 5, 20, 150 sweeps = {relax[rule][[4, 19, 149]].round(3)};  "
          f"equilibrium e at T = 2.8: {study.mean:.4f} +/- {study.standard_error:.4f}")
rule_gap = abs(equilibrium["metropolis"].mean - equilibrium["glauber"].mean)
rule_sigma = np.hypot(equilibrium["metropolis"].standard_error,
                      equilibrium["glauber"].standard_error)
print(f"difference of equilibria: {rule_gap:.4f} = {rule_gap / rule_sigma:.1f} combined sigma")

כלל גלאובר מגיע לרלקסציה לאט יותר — הוא מקבל כל מהלך בתדירות נמוכה יותר — ושניהם מגיעים
לאותו שיווי משקל. "סריקה" היא יחידה של חישוב, לא של זמן: שום קבוע פיזיקלי אינו ממיר אותה
לשניות, ושיטת מונטה קרלו אינה מראה לכם כיצד מגנט ממשי נע.

## חלק 9 — מדידה: טמפרטורת המעבר

אתרו את שיא הסוספטיביליות לכל גודל בדיוק רב יותר ממה שהרשת של חלק 4 מאפשרת, על ידי התאמת
פרבולה דרך שלוש הנקודות הגבוהות ביותר, ואקסטרפלו את מיקומי השיא אל $L \to \infty$ בהנחה
ש-$T_{\mathrm{peak}}(L) = T_c + a/L$ (סקיילינג בגודל סופי עם המעריך המדויק
$\nu = 1$, מצוטט, לא נגזר). אחר כך בדקו את השיא של $L = 16$ תחת שלושה זרעים נוספים.

### לנבא

האם $T_c$ המאוקסטרפל ינחת בטווח $0.05$ מהערך $2.269$ של אונסאגר? האם ארבעת מיקומי השיא
של $L = 16$ יסכימו ביניהם טוב יותר מ-$0.05$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
def parabola_peak(t, y):
    # Vertex of the parabola through the highest point and its two neighbours.
    k = int(np.clip(np.argmax(y), 1, len(y) - 2))
    coeffs = np.polyfit(t[k - 1:k + 2], y[k - 1:k + 2], 2)
    return float(-coeffs[1] / (2 * coeffs[0]))


refined = {size: parabola_peak(temps, column(size, "susceptibility")) for size in SIZES}
inverse_l = np.array([1 / s for s in SIZES])
slope, t_c_estimate = np.polyfit(inverse_l, [refined[s] for s in SIZES], 1)

# Seed check at L = 16: three more independent scans near the peak.
near = np.array([2.6, 2.55, 2.5, 2.46, 2.42, 2.38, 2.34, 2.3, 2.26, 2.22])
seed_peaks = [refined[16]]
for s in np.random.SeedSequence(9).spawn(3):
    scan = ising.temperature_scan(16, near, 1200, np.random.default_rng(s), burn_in=300)
    seed_peaks.append(parabola_peak(near, np.array([r.susceptibility for r in scan])))
seed_spread = float(np.std(seed_peaks, ddof=1))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(inverse_l, [refined[s] for s in SIZES], "o", color="#dc2626", ms=7,
        label="chi' peak")
x = np.linspace(0, 0.13, 50)
ax.plot(x, t_c_estimate + slope * x, "--", color="k", label="T_c + a/L")
ax.axhline(T_C, color="grey", ls=":", label="Onsager")
ax.set_xlabel("1 / L")
ax.set_ylabel("T_peak")
ax.legend()
plt.tight_layout()
plt.show()

for size in SIZES:
    print(f"L = {size:2d}: chi' peak at T = {refined[size]:.3f}")
print(f"extrapolated T_c = {t_c_estimate:.3f};  Onsager {T_C:.3f};  "
      f"difference {t_c_estimate - T_C:+.3f}")
print(f"L = 16 peak over four seeds: {np.round(seed_peaks, 3)}, spread {seed_spread:.3f}")

התוצאה המצוטטת היא $T_c$ = (הערך המאוקסטרפל) $\pm$ (אי-ודאות בגודל הפיזור בין הזרעים
של $L = 16$, מוגדלת עבור אקסטרפולציה שנשענת על שלושה סריגים קטנים) — ושתי הסתייגויות ששום
פס שגיאה אינו לוכד: האקסטרפולציה *מניחה* $\nu = 1$, והשיא של
$\chi'$ הוא אחת מכמה הגדרות בגודל סופי של "טמפרטורת המעבר", שכל אחת מהן נסחפת אחרת. מה
שהמדידה מבססת הוא מגמה שמצביעה על $2.27$; היא אינה מראה, ולא יכולה להראות, שמשהו סינגולרי
שם. זו טענה על $N \to \infty$, שאף סימולציה אינה מגיעה אליו.

## חלק 10 — בדיקות אוטומטיות

חישוב שלא בדקתם הוא תמונה, לא ראיה. אלה אותן טענות שרצות בחבילת הבדיקות של הפרויקט.

In [ ]:
# 1. Bookkeeping: the incrementally updated totals equal a from-scratch recount.
state = ising.random_state(16, rng)
_, _, state = ising.simulate(state, 2.3, 500, rng, burn_in=0)
assert state.energy == ising.lattice_energy(state.lattice)
assert state.magnetization == int(state.lattice.sum())

# 2. The chain reproduces its exact solution within four standard errors (Part 3).
for (temperature, field), results in chain.items():
    e_exact = ising.ising_1d_exact(temperature, field)[0]
    assert abs(results[:, 0].mean() - e_exact) < 4 * results[:, 0].std(ddof=1) / 2

# 3. Ordered below, disordered above; the deep-phase values match Onsager-Yang (Part 4).
for size in SIZES:
    m_abs = column(size, "abs_magnetization")
    assert m_abs[-1] > 0.95 and m_abs[0] < 0.35
assert relative_error(column(32, "abs_magnetization")[-1], ising.onsager_magnetization(1.8)) < 0.01

# 4. The susceptibility peak grows with L and sits above T_c, closer for larger L (Parts 4, 9).
heights = [column(size, "susceptibility").max() for size in SIZES]
assert heights[0] < heights[1] < heights[2]
assert refined[8] > refined[32] > T_C - 0.05

# 5. Critical slowing down: tau peaks near T_c and grows with L (Part 5).
for size in SIZES:
    tau = column(size, "tau_abs_magnetization")
    assert 2.15 < temps[np.argmax(tau)] < 2.6
assert tau_peak[8] < tau_peak[32]

# 6. The small lattice reverses, the large one does not (Part 6).
assert ising.branch_flips(branch[8]) >= 2 and ising.branch_flips(branch[32]) == 0

# 7. Below T_c the loop stays open; above it, it nearly closes (Part 7).
assert loops[(1.5, 40)][2] > 1.0 and loops[(3.0, 10)][2] < 0.3

# 8. Two rules, one equilibrium (Part 8).
assert rule_gap < 3 * rule_sigma

print("all checks passed")

## חלק 11 — חקרו בעצמכם

בחרו את הגודל, הטמפרטורה, השדה, מצב ההתחלה, כלל העדכון ואורך הריצה, ואז לחצו על
**Run Interact**. משמאל: התצורה הסופית. מימין: העקבות של $m$ ושל $e$, עם זמן המתאם העצמי של
כל אחת. דברים לנסות: צינון חטוף אל $T = 1.0$ מהתחלה חמה ב-$L = 64$;
$T = 2.27$ ב-$L = 16$ וב-$L = 64$ עם אותו אורך ריצה, תוך השוואת $\tau$; שדה שלילי קטן על
סריג שכולו למעלה ב-$T = 1.5$; כלל גלאובר מול אלגוריתם מטרופוליס ב-$T = 2.27$.

In [ ]:
import ipywidgets as widgets


def explore(size=32, temperature=2.27, field=0.0, start="hot", rule="metropolis", sweeps=1000):
    gen = np.random.default_rng(0)
    state = (ising.random_state(size, gen, field=field) if start == "hot"
             else ising.with_field(ising.aligned_state(size), field))
    m, e, state = ising.simulate(state, temperature, sweeps, gen, burn_in=0, rule=rule)
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5),
                                      gridspec_kw={"width_ratios": [1, 1.6]})
    left.imshow(state.lattice, cmap="Blues", vmin=-1.5, vmax=1)
    left.set_xticks([])
    left.set_yticks([])
    left.set_title(f"L = {size}, T = {temperature}, h = {field}")
    right.plot(m, lw=0.8, label="m")
    right.plot(e, lw=0.8, label="E / (N J)")
    right.set_xlabel("sweep")
    right.legend()
    plt.tight_layout()
    plt.show()
    half = m[len(m) // 2:]
    print(f"last half of the run: <m> = {half.mean():+.3f}, <|m|> = {np.abs(half).mean():.3f}, "
          f"tau(|m|) = {ising.autocorrelation_time(np.abs(half)):.1f} sweeps, "
          f"tau(e) = {ising.autocorrelation_time(e[len(e) // 2:]):.1f} sweeps")
    print(f"Onsager-Yang |m| at this T: {ising.onsager_magnetization(temperature):.3f}")


widgets.interact_manual(
    explore,
    size=widgets.Dropdown(options=[16, 32, 64], value=32),
    temperature=widgets.FloatSlider(min=1.0, max=4.0, step=0.01, value=2.27),
    field=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0),
    start=widgets.ToggleButtons(options=["hot", "cold"]),
    rule=widgets.ToggleButtons(options=["metropolis", "glauber"]),
    sweeps=widgets.IntSlider(min=100, max=4000, step=100, value=1000),
)

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון בבדיקה אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "15-ising.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שתלכו

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהנמקה שלכם?
2. בחלק 6 שני הסריגים הודמו מאותן משוואות באותה טמפרטורה. הסבירו במילים שלכם מדוע אחד מהם
   "בחר" כיוון והשני לא.
3. המדידה שלכם בחלק 9 מצטטת מספר עם אי-ודאות. מנו כל הנחה שהמספר הזה נשען עליה ושפס השגיאה
   שלו *אינו* כולל.
4. מדוע שום סימולציה, ארוכה ככל שתהיה, אינה יכולה להוכיח שלסריג הריבועי יש מעבר פאזה?

**התשובות שלכם:**

1.
2.
3.
4.